# **X API Scraper Notebook for HealthPH+**


This notebook collects posts from X (Twitter) using the X API v2 recent search endpoint and filters by health keywords.

### Requirements
- X Developer app with API access
- Bearer token stored in environment variable: `X_BEARER_TOKEN`
- `.env` file is optional (loaded automatically if present)


# **Dependencies**


In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv


In [ ]:
def _find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "data").exists():
            return candidate
    return current


PROJECT_ROOT = _find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv()

print(f"Project root: {PROJECT_ROOT}")
print("Libraries loaded successfully")


## **Configuration**


In [ ]:
# X API configuration
X_BEARER_TOKEN = os.getenv("X_BEARER_TOKEN", "").strip()

# Full-archive endpoint (requires appropriate X API access level)
X_SEARCH_ENDPOINT = "https://api.x.com/2/tweets/search/all"
X_MAX_RESULTS = 100                 # all-search supports 10-500
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 1.0

# Search date range parameters (date from / until)
# Accepts values parseable by pandas, e.g. "2026-02-01" or "2026-02-01 08:00:00+08:00"
SEARCH_DATE_FROM = None
SEARCH_DATE_UNTIL = None

# Budget controls
TOTAL_POST_BUDGET = 15000
PER_KEYWORD_TARGET = None           # None -> auto floor(TOTAL_POST_BUDGET / total_keywords)
MAX_EMPTY_OR_DUPLICATE_PAGES = 2    # Stop paging a keyword after this many low-yield pages

# Resume controls
RESUME_FROM_STATE = True
STATE_FILE = PROJECT_ROOT / "data/raw/twitter/x_api_state.json"
LOAD_EXISTING_IDS_FROM_OUTPUT = True

# Query behavior
INCLUDE_REPLIES = False
INCLUDE_RETWEETS = False

# Keyword source (all keyword docs)
KEYWORD_FILES = [
    PROJECT_ROOT / "docs/keywords/covid_keywords.csv",
    PROJECT_ROOT / "docs/keywords/covid_keywords2.csv",
    PROJECT_ROOT / "docs/keywords/ri_keywords.csv",
    PROJECT_ROOT / "docs/keywords/ri_keywords2.csv",
    PROJECT_ROOT / "docs/keywords/tb_keywords.csv",
    PROJECT_ROOT / "docs/keywords/tb_keywords2.csv",
    PROJECT_ROOT / "docs/keywords/pn_keywords.csv",
    PROJECT_ROOT / "docs/keywords/keywords_master.csv",
]
KEYWORD_COLUMNS = [
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    "symptoms",  # keywords_master.csv has disease + symptoms columns
]

MANUAL_KEYWORDS = [
    # "ubo",
    # "sipon",
]

OUTPUT_FILE = PROJECT_ROOT / "data/raw/twitter/x_api_results.csv"

print("X endpoint:", X_SEARCH_ENDPOINT)
print("Output file:", OUTPUT_FILE)
print("State file:", STATE_FILE)
print("Bearer token loaded:", bool(X_BEARER_TOKEN))
print("Search date from:", SEARCH_DATE_FROM)
print("Search date until:", SEARCH_DATE_UNTIL)
print("Total post budget:", TOTAL_POST_BUDGET)


## **Helper Functions**


In [ ]:
def load_keywords_from_csv(filepath: Path, column: str | None = None) -> list[str]:
    if not filepath.exists():
        raise FileNotFoundError(f"Keyword file not found: {filepath}")

    df = pd.read_csv(filepath)
    if df.empty:
        return []

    selected_col = column if (column and column in df.columns) else df.columns[0]
    values = df[selected_col].dropna().astype(str)

    keywords: list[str] = []
    for raw in values:
        chunks = [part.strip().strip('"').strip("'") for part in raw.split(",")]
        keywords.extend([chunk for chunk in chunks if chunk])

    return list(dict.fromkeys(keywords))


def load_keywords_from_files(files: list[Path], columns: list[str | None]) -> list[str]:
    if len(files) != len(columns):
        raise ValueError("files and columns must have the same length")

    merged: list[str] = []
    for filepath, column in zip(files, columns):
        merged.extend(load_keywords_from_csv(filepath=filepath, column=column))

    return list(dict.fromkeys(merged))


def to_rfc3339_utc(value: str | None) -> str | None:
    if value is None:
        return None

    raw = str(value).strip()
    if not raw:
        return None

    parsed = pd.to_datetime(raw, utc=True, errors="coerce")
    if pd.isna(parsed):
        raise ValueError(
            f"Invalid date/time value: {value}. Use a pandas-parseable format like '2026-02-01' or RFC3339."
        )

    return parsed.strftime("%Y-%m-%dT%H:%M:%SZ")


def build_x_query(keyword: str, include_replies: bool = False, include_retweets: bool = False) -> str:
    term = keyword.strip()
    if not term:
        raise ValueError("Keyword must not be empty")

    escaped = term.replace('"', r'\"')
    tokens = [f'"{escaped}"']

    if not include_replies:
        tokens.append("-is:reply")
    if not include_retweets:
        tokens.append("-is:retweet")

    return " ".join(tokens)


def _extract_metrics(tweet: dict[str, Any]) -> dict[str, Any]:
    metrics = tweet.get("public_metrics") or {}
    return {
        "like_count": metrics.get("like_count"),
        "comment_count": metrics.get("reply_count"),
        "share_count": metrics.get("retweet_count"),
        "quote_count": metrics.get("quote_count"),
        "impression_count": metrics.get("impression_count"),
        "bookmark_count": metrics.get("bookmark_count"),
    }


def _resolve_max_results(endpoint: str, requested_max_results: int) -> int:
    max_allowed = 100 if endpoint.rstrip("/").endswith("/recent") else 500
    return max(10, min(int(requested_max_results), max_allowed))


def _keyword_set_hash(keywords: list[str]) -> str:
    normalized = [k.strip().lower() for k in keywords if str(k).strip()]
    joined = "\n".join(sorted(dict.fromkeys(normalized)))
    return hashlib.sha1(joined.encode("utf-8")).hexdigest()


def load_existing_ids_from_csv(filepath: Path) -> set[str]:
    if not filepath.exists():
        return set()

    try:
        df = pd.read_csv(filepath, usecols=["id"])
    except Exception as exc:
        print(f"Warning: could not read existing ids from {filepath}: {exc}")
        return set()

    return set(df["id"].dropna().astype(str).str.strip())


def load_scrape_state(state_file: Path, keyword_hash: str) -> dict[str, Any]:
    if not state_file.exists():
        return {}

    try:
        payload = json.loads(state_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Warning: could not parse state file {state_file}: {exc}")
        return {}

    if payload.get("keyword_set_hash") != keyword_hash:
        print("State keyword set mismatch. Ignoring old pagination state.")
        return {}

    return payload


def save_scrape_state(state_file: Path, payload: dict[str, Any], keyword_hash: str) -> None:
    state_file.parent.mkdir(parents=True, exist_ok=True)

    out = dict(payload)
    out["keyword_set_hash"] = keyword_hash
    out["saved_at_utc"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")

    state_file.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")


def build_keyword_matcher(keywords: list[str]) -> list[tuple[str, str]]:
    return [(kw, kw.lower()) for kw in keywords if kw]


def match_keywords_in_text(text: str, matcher: list[tuple[str, str]]) -> list[str]:
    hay = str(text or "").lower()
    if not hay:
        return []

    matches: list[str] = []
    for original, needle in matcher:
        if needle and needle in hay:
            matches.append(original)
    return matches


def fetch_x_page_for_keyword(
    keyword: str,
    bearer_token: str,
    endpoint: str,
    max_results: int = 100,
    start_time: str | None = None,
    end_time: str | None = None,
    include_replies: bool = False,
    include_retweets: bool = False,
    timeout_seconds: int = 30,
    next_token: str | None = None,
) -> tuple[pd.DataFrame, str | None]:
    if not bearer_token:
        raise ValueError("Missing bearer token. Set X_BEARER_TOKEN in your environment or .env file.")

    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "User-Agent": "HealthPHXAPICollector/1.0",
    }

    query = build_x_query(
        keyword=keyword,
        include_replies=include_replies,
        include_retweets=include_retweets,
    )

    params: dict[str, Any] = {
        "query": query,
        "max_results": _resolve_max_results(endpoint=endpoint, requested_max_results=max_results),
        "tweet.fields": "id,text,created_at,lang,author_id,public_metrics",
    }
    if start_time:
        params["start_time"] = start_time
    if end_time:
        params["end_time"] = end_time
    if next_token:
        params["next_token"] = next_token

    response = requests.get(endpoint, headers=headers, params=params, timeout=timeout_seconds)

    if response.status_code != 200:
        snippet = response.text[:500]
        hint = ""
        if response.status_code == 403:
            hint = " Check your X API plan/access for this endpoint (search/all requires full-archive access)."
        if response.status_code == 429:
            hint = " Rate limit reached. Lower request volume or add longer delays."
        raise RuntimeError(
            f"X API request failed for keyword '{keyword}'. "
            f"HTTP {response.status_code}: {snippet}{hint}"
        )

    payload = response.json()
    tweets = payload.get("data", [])
    meta = payload.get("meta", {})
    next_token_out = meta.get("next_token")

    rows: list[dict[str, Any]] = []
    for tweet in tweets:
        tweet_id = str(tweet.get("id", "")).strip()
        if not tweet_id:
            continue

        row = {
            "created_at": tweet.get("created_at", ""),
            "id": f"tweet-{tweet_id}",
            "tweet_id": tweet_id,
            "text": tweet.get("text", ""),
            "keyword": keyword,
            "source": "twitter",
            "lang": tweet.get("lang", ""),
            "author_id": tweet.get("author_id", ""),
            "url": f"https://x.com/i/web/status/{tweet_id}",
            "scraped_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC"),
        }
        row.update(_extract_metrics(tweet))
        rows.append(row)

    columns = [
        "created_at",
        "id",
        "tweet_id",
        "text",
        "keyword",
        "source",
        "lang",
        "author_id",
        "url",
        "like_count",
        "comment_count",
        "share_count",
        "quote_count",
        "impression_count",
        "bookmark_count",
        "scraped_at_utc",
    ]

    if not rows:
        return pd.DataFrame(columns=columns), next_token_out

    return pd.DataFrame(rows, columns=columns), next_token_out


def scrape_x_keywords_with_budget(
    keywords: list[str],
    bearer_token: str,
    endpoint: str,
    total_post_budget: int = 15000,
    per_keyword_target: int | None = None,
    max_results: int = 100,
    max_empty_or_duplicate_pages: int = 2,
    start_time: str | None = None,
    end_time: str | None = None,
    include_replies: bool = False,
    include_retweets: bool = False,
    timeout_seconds: int = 30,
    sleep_seconds: float = 1.0,
    initial_seen_ids: set[str] | None = None,
    initial_next_tokens: dict[str, str | None] | None = None,
    initial_empty_streak: dict[str, int] | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    clean_keywords = [str(k).strip() for k in keywords if str(k).strip()]
    clean_keywords = list(dict.fromkeys(clean_keywords))
    if not clean_keywords:
        return pd.DataFrame(), {}

    budget = max(1, int(total_post_budget))
    if per_keyword_target is None:
        target = max(1, budget // len(clean_keywords))
    else:
        target = max(1, int(per_keyword_target))

    matcher = build_keyword_matcher(clean_keywords)

    seen_ids = set(initial_seen_ids or set())
    next_tokens = {kw: None for kw in clean_keywords}
    if initial_next_tokens:
        for kw, token in initial_next_tokens.items():
            if kw in next_tokens:
                next_tokens[kw] = token

    empty_streak = {kw: 0 for kw in clean_keywords}
    if initial_empty_streak:
        for kw, value in initial_empty_streak.items():
            if kw in empty_streak:
                try:
                    empty_streak[kw] = int(value)
                except Exception:
                    empty_streak[kw] = 0

    rows_by_id: dict[str, dict[str, Any]] = {}
    keyword_post_ids = {kw: set() for kw in clean_keywords}
    active_keywords = set(clean_keywords)

    round_index = 0
    while active_keywords and len(rows_by_id) < budget:
        round_index += 1
        progress_this_round = False

        for kw in clean_keywords:
            if kw not in active_keywords:
                continue
            if len(rows_by_id) >= budget:
                break

            if len(keyword_post_ids[kw]) >= target:
                active_keywords.discard(kw)
                continue

            df_page, next_token_out = fetch_x_page_for_keyword(
                keyword=kw,
                bearer_token=bearer_token,
                endpoint=endpoint,
                max_results=max_results,
                start_time=start_time,
                end_time=end_time,
                include_replies=include_replies,
                include_retweets=include_retweets,
                timeout_seconds=timeout_seconds,
                next_token=next_tokens.get(kw),
            )
            next_tokens[kw] = next_token_out

            new_count = 0
            for record in df_page.to_dict(orient="records"):
                post_id = str(record.get("id", "")).strip()
                if not post_id or post_id in seen_ids:
                    continue

                matched_keywords = match_keywords_in_text(record.get("text", ""), matcher)
                if not matched_keywords:
                    matched_keywords = [kw]

                seen_ids.add(post_id)
                record["matched_keywords"] = set(matched_keywords)
                rows_by_id[post_id] = record
                progress_this_round = True
                new_count += 1

                for matched in matched_keywords:
                    if matched in keyword_post_ids:
                        keyword_post_ids[matched].add(post_id)

                if len(rows_by_id) >= budget:
                    break

            if new_count == 0:
                empty_streak[kw] += 1
            else:
                empty_streak[kw] = 0

            exhausted = (
                len(keyword_post_ids[kw]) >= target
                or not next_tokens.get(kw)
                or empty_streak[kw] >= max_empty_or_duplicate_pages
            )
            if exhausted:
                active_keywords.discard(kw)

            print(
                f"Round {round_index} | Keyword '{kw}' | new={new_count} | "
                f"coverage={len(keyword_post_ids[kw])}/{target} | "
                f"global={len(rows_by_id)}/{budget}"
            )

            if sleep_seconds > 0 and len(rows_by_id) < budget:
                time.sleep(sleep_seconds)

        if not progress_this_round:
            break

    if not rows_by_id:
        summary = {
            "budget": budget,
            "per_keyword_target": target,
            "collected": 0,
            "remaining_keywords": sorted(active_keywords),
            "next_tokens": next_tokens,
            "empty_streak": empty_streak,
            "keyword_counts": {kw: 0 for kw in clean_keywords},
            "rounds": round_index,
        }
        return pd.DataFrame(), summary

    out = pd.DataFrame(rows_by_id.values())
    out["matched_keywords"] = out["matched_keywords"].apply(lambda values: ", ".join(sorted(values)))
    out = out.sort_values(by=["created_at", "id"], ascending=[False, True], kind="stable").reset_index(drop=True)

    summary = {
        "budget": budget,
        "per_keyword_target": target,
        "collected": int(len(out)),
        "remaining_keywords": sorted(active_keywords),
        "next_tokens": next_tokens,
        "empty_streak": empty_streak,
        "keyword_counts": {kw: len(ids) for kw, ids in keyword_post_ids.items()},
        "rounds": round_index,
    }
    return out, summary


def save_results(df: pd.DataFrame, filepath: Path) -> None:
    filepath.parent.mkdir(parents=True, exist_ok=True)

    if df.empty:
        print("No rows to save.")
        return

    to_save = df.copy()
    to_save["id"] = to_save["id"].astype(str).str.strip()
    to_save = to_save[to_save["id"] != ""]
    to_save = to_save.drop_duplicates(subset=["id"], keep="first")

    file_exists = filepath.exists()
    incoming_count = len(to_save)

    if file_exists:
        try:
            existing_ids = set(
                pd.read_csv(filepath, usecols=["id"])["id"].dropna().astype(str).str.strip()
            )
            to_save = to_save[~to_save["id"].isin(existing_ids)]
        except Exception as exc:
            print(f"Warning: could not load existing ids for dedupe: {exc}")

    if to_save.empty:
        print("No new rows to append after dedupe.")
        return

    to_save.to_csv(
        filepath,
        mode="a" if file_exists else "w",
        index=False,
        header=not file_exists,
        encoding="utf-8-sig",
    )

    print(
        f"Saved {len(to_save)} new rows to {filepath} "
        f"(incoming: {incoming_count}, skipped: {incoming_count - len(to_save)})"
    )


print("Helper functions defined")


## **Run Scraper**


In [ ]:
# Build keyword list
file_keywords = load_keywords_from_files(KEYWORD_FILES, KEYWORD_COLUMNS) if KEYWORD_FILES else []
manual_keywords = [k.strip() for k in MANUAL_KEYWORDS if str(k).strip()]
keywords = list(dict.fromkeys([*file_keywords, *manual_keywords]))

search_from = to_rfc3339_utc(SEARCH_DATE_FROM)
search_until = to_rfc3339_utc(SEARCH_DATE_UNTIL)
if search_from and search_until and search_from >= search_until:
    raise ValueError("SEARCH_DATE_FROM must be earlier than SEARCH_DATE_UNTIL.")

if not X_BEARER_TOKEN:
    raise ValueError(
        "X_BEARER_TOKEN is not set. Add it to your environment or .env file, then rerun this cell."
    )

keyword_hash = _keyword_set_hash(keywords)
state_payload = {}
if RESUME_FROM_STATE:
    state_payload = load_scrape_state(STATE_FILE, keyword_hash=keyword_hash)

existing_ids = set()
if LOAD_EXISTING_IDS_FROM_OUTPUT:
    existing_ids = load_existing_ids_from_csv(OUTPUT_FILE)

initial_next_tokens = state_payload.get("next_tokens", {}) if state_payload else {}
initial_empty_streak = state_payload.get("empty_streak", {}) if state_payload else {}

print(f"Loaded {len(keywords)} unique keyword(s)")
print("Sample:", keywords[:10])
print("Search window:", search_from or "open", "->", search_until or "open")
print("Existing ids loaded:", len(existing_ids))

x_df, scrape_summary = scrape_x_keywords_with_budget(
    keywords=keywords,
    bearer_token=X_BEARER_TOKEN,
    endpoint=X_SEARCH_ENDPOINT,
    total_post_budget=TOTAL_POST_BUDGET,
    per_keyword_target=PER_KEYWORD_TARGET,
    max_results=X_MAX_RESULTS,
    max_empty_or_duplicate_pages=MAX_EMPTY_OR_DUPLICATE_PAGES,
    start_time=search_from,
    end_time=search_until,
    include_replies=INCLUDE_REPLIES,
    include_retweets=INCLUDE_RETWEETS,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    sleep_seconds=REQUEST_SLEEP_SECONDS,
    initial_seen_ids=existing_ids,
    initial_next_tokens=initial_next_tokens,
    initial_empty_streak=initial_empty_streak,
)

if RESUME_FROM_STATE:
    save_scrape_state(STATE_FILE, payload=scrape_summary, keyword_hash=keyword_hash)

print("Total new unique posts collected:", len(x_df))
print("Rounds:", scrape_summary.get("rounds"))
print("Remaining keywords:", len(scrape_summary.get("remaining_keywords", [])))

coverage = (
    pd.Series(scrape_summary.get("keyword_counts", {}), name="new_unique_posts")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "keyword"})
)
coverage.head(20)


## **Save to CSV**


In [ ]:
save_results(x_df, OUTPUT_FILE)
x_df.sample(min(10, len(x_df))) if not x_df.empty else x_df
